# Aula 23 — Treino e depuração de uma MLP

NumPy puro, sem autograd. Python >=3.11, NumPy >=1.26 e Matplotlib >=3.8.
Seed principal: 20260923. Dados sintéticos; cada linha é uma observação independente.
Execute todas as células em ordem. O teste é consultado somente após a seleção na validação.
As figuras são geradas em memória; não há downloads nem arquivos auxiliares obrigatórios.


In [ ]:
import copy
import sys
import warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
warnings.simplefilter("error")
SEED = 20260923
assert sys.version_info >= (3, 11)
assert np.lib.NumpyVersion(np.__version__) >= "1.26.0"
print("Python", sys.version.split()[0], "NumPy", np.__version__, "Matplotlib", matplotlib.__version__)


## 1. Dados, IDs e split antes do pré-processamento

Três classes gaussianas parcialmente sobrepostas, 240 observações por classe.
As duas features têm escalas distintas. O split estratificado separa 144/48/48 exemplos
por classe em treino/validação/teste. A estratificação é adequada aqui porque as linhas
são independentes; em dados reais com pessoas, equipamentos ou tempo, o split deve
respeitar essas unidades. Os rótulos são 0, 1 e 2, sem ordem ordinal.


In [ ]:
data_rng = np.random.default_rng(SEED)
centers = np.array([[-1.5,-0.8], [1.5,-0.8], [0.,1.5]])
Xraw = np.vstack([data_rng.normal(c, 0.95, (240,2)) for c in centers]) * [1.,100.]
y = np.repeat(np.arange(3),240)
ids = np.arange(len(y))
split_rng = np.random.default_rng(SEED+1)
tr,va,te = [],[],[]
for c in range(3):
    ix = split_rng.permutation(ids[y==c])
    tr.extend(ix[:144]); va.extend(ix[144:192]); te.extend(ix[192:])
tr,va,te = map(np.array,(tr,va,te))
assert set(tr).isdisjoint(va) and set(tr).isdisjoint(te) and set(va).isdisjoint(te)
assert len(set(tr)|set(va)|set(te)) == len(y)
mu, scale = Xraw[tr].mean(0), Xraw[tr].std(0)
assert np.all(scale>0)
X = (Xraw-mu)/scale
assert np.max(np.abs(X[tr].mean(0))) < 1e-12
assert np.isfinite(X).all() and y.dtype.kind in "iu"
print("splits", len(tr),len(va),len(te), "classes de treino", np.bincount(y[tr]))


## 2. MLP com opções explícitas

Arquitetura 2 → H → 3, tanh, Xavier normal e logits sem ativação final.
LayerNorm opcional opera antes da tanh; dropout invertido opera depois dela.
A configuração mínima desliga ambos. Em eval não há máscara nem consumo de RNG.
LayerNorm não precisa de médias correntes. BatchNorm fica como extensão da Aula 22.


In [ ]:
def init(seed, h=16, ln=False):
    r=np.random.default_rng(seed)
    p={"W1":r.normal(0,np.sqrt(2/(2+h)),(2,h)), "b1":np.zeros((1,h)),
       "W2":r.normal(0,np.sqrt(2/(h+3)),(h,3)), "b2":np.zeros((1,3))}
    if ln: p.update(gamma=np.ones((1,h)), beta=np.zeros((1,h)))
    return p

def forward(p,x,training=False,q=1.,rng=None):
    assert x.ndim==2 and x.shape[1]==p["W1"].shape[0]
    assert 0<q<=1
    z=x@p["W1"]+p["b1"]
    if "gamma" in p:
        inv=1/np.sqrt(z.var(1,keepdims=True)+1e-5)
        zh=(z-z.mean(1,keepdims=True))*inv
        u=p["gamma"]*zh+p["beta"]
    else: inv=zh=None; u=z
    a=np.tanh(u)
    mask=np.ones_like(a)
    if training and q<1:
        assert rng is not None
        mask=(rng.random(a.shape)<q)/q
    ad=a*mask
    logits=ad@p["W2"]+p["b2"]
    return logits,(x,a,ad,mask,zh,inv)

def ce(logits,labels):
    assert labels.shape==(len(logits),) and labels.dtype.kind in "iu"
    assert len(labels)>0 and np.all((labels>=0)&(labels<logits.shape[1]))
    shifted=logits-logits.max(1,keepdims=True)
    logp=shifted-np.log(np.exp(shifted).sum(1,keepdims=True))
    return float(-logp[np.arange(len(labels)),labels].mean()),np.exp(logp)

def loss_grad(p,x,labels,lam=0.,training=False,q=1.,rng=None):
    logits,cache=forward(p,x,training,q,rng)
    loss,prob=ce(logits,labels)
    penalty=lam/2*(np.sum(p["W1"]**2)+np.sum(p["W2"]**2))
    dz=prob.copy(); dz[np.arange(len(labels)),labels]-=1; dz/=len(labels)
    x,a,ad,mask,zh,inv=cache
    g={"W2":ad.T@dz+lam*p["W2"], "b2":dz.sum(0,keepdims=True)}
    du=(dz@p["W2"].T)*mask*(1-a*a)
    if "gamma" in p:
        g["gamma"]=(du*zh).sum(0,keepdims=True); g["beta"]=du.sum(0,keepdims=True)
        d=du*p["gamma"]
        du=inv*(d-d.mean(1,keepdims=True)-zh*(d*zh).mean(1,keepdims=True))
    g["W1"]=x.T@du+lam*p["W1"]; g["b1"]=du.sum(0,keepdims=True)
    assert set(g)==set(p)
    assert all(g[k].shape==p[k].shape and np.isfinite(g[k]).all() for k in p)
    return loss+penalty,g,cache


## 3. Sanidade antes de treinar

Logits iguais produzem CE = ln(3). Permutar ou duplicar linhas não altera a loss média
nem o gradiente agregado quando dropout está desligado e não há BatchNorm.
O teste negativo injeta labels com shape inválido e verifica sua rejeição.


In [ ]:
p0=init(SEED+2)
xb,yb=X[tr[:9]],y[tr[:9]]
uniform,_=ce(np.zeros((9,3)),yb)
assert abs(uniform-np.log(3))<1e-14
loss0,g0,_=loss_grad(p0,xb,yb)
perm=np.arange(9)[::-1]
lp,gp,_=loss_grad(p0,xb[perm],yb[perm])
ld,gd,_=loss_grad(p0,np.vstack([xb,xb]),np.r_[yb,yb])
assert abs(loss0-lp)<1e-14 and abs(loss0-ld)<1e-14
assert all(np.allclose(g0[k],gp[k]) and np.allclose(g0[k],gd[k]) for k in p0)
rejected=False
try: ce(np.zeros((9,3)),yb[:,None])
except AssertionError: rejected=True
assert rejected
print("CE uniforme",uniform,"; shapes, permutação e duplicação: OK")


## 4. Gradiente numérico e defeito injetado

Uma rede pequena com LayerNorm e L2 exercita todos os parâmetros. A máscara dropout
é congelada recriando o RNG em cada chamada. O defeito divide dW1 uma segunda vez
pelo tamanho do lote. O check deve aprovar a implementação e reprovar esse defeito.


In [ ]:
pc=init(SEED+3,h=4,ln=True)
xc=np.array([[.2,-.7],[1.1,.4],[-.8,.9]])
yc=np.array([0,1,2])
def obj(): return loss_grad(pc,xc,yc,lam=.03,training=True,q=.8,rng=np.random.default_rng(75))[0]
_,ga,_=loss_grad(pc,xc,yc,lam=.03,training=True,q=.8,rng=np.random.default_rng(75))
gn={k:np.zeros_like(v) for k,v in pc.items()}
for k in pc:
    for ix in np.ndindex(pc[k].shape):
        old=pc[k][ix]; h=1e-5
        pc[k][ix]=old+h; plus=obj()
        pc[k][ix]=old-h; minus=obj()
        pc[k][ix]=old
        gn[k][ix]=(plus-minus)/(2*h)
errors={k:float(np.linalg.norm(ga[k]-gn[k])/max(1e-12,np.linalg.norm(ga[k])+np.linalg.norm(gn[k]))) for k in pc}
bad=ga["W1"]/len(yc)
bad_error=np.linalg.norm(bad-gn["W1"])/(np.linalg.norm(bad)+np.linalg.norm(gn["W1"]))
assert max(errors.values())<1e-7
assert bad_error>.1
print("erros relativos por tensor",errors,"; defeito",bad_error)


## 5. Atualização atômica e instrumentação

Momentum usa v ← beta × v + g; theta ← theta − eta × v. Todos os gradientes vêm
dos mesmos pesos antigos. A razão atualização/peso é apenas diagnóstico, sem limiar
universal. Vieses inicialmente nulos tornam essa razão pouco informativa.


In [ ]:
def step(p,g,v,eta,beta):
    assert eta>=0 and 0<=beta<1
    new_v={k:beta*v[k]+g[k] for k in p}
    new_p={k:p[k]-eta*new_v[k] for k in p}
    assert all(np.isfinite(a).all() for a in new_p.values())
    ratio={k:float(np.linalg.norm(new_p[k]-p[k])/max(1e-12,np.linalg.norm(p[k]))) for k in ("W1","W2")}
    return new_p,new_v,ratio

v0={k:np.zeros_like(a) for k,a in p0.items()}
p1,_,_=step(p0,g0,v0,1e-3,0.)
assert loss_grad(p1,xb,yb)[0]<loss0
frozen,_,_=step(p0,g0,v0,0.,0.)
assert all(np.array_equal(p0[k],frozen[k]) for k in p0)
assert any(np.linalg.norm(g0[k])>0 for k in g0)
print("passo pequeno reduz loss; eta=0 detectado como ausência de atualização")


## 6. Overfit de um lote fixo

Selecionamos quatro observações de cada classe do treino. Desligamos regularização,
dropout e normalização. Conseguir memorizar este lote é uma condição de sanidade,
sem demonstrar generalização. Labels contraditórios em entradas idênticas têm outro limite.


In [ ]:
tiny=np.concatenate([tr[y[tr]==c][:4] for c in range(3)])
pt=init(SEED+4,h=32); vt={k:np.zeros_like(v) for k,v in pt.items()}
tiny_initial=loss_grad(pt,X[tiny],y[tiny])[0]
for _ in range(2500):
    _,g,_=loss_grad(pt,X[tiny],y[tiny])
    pt,vt,_=step(pt,g,vt,.05,.9)
tiny_loss,prob=ce(forward(pt,X[tiny])[0],y[tiny])
tiny_acc=np.mean(prob.argmax(1)==y[tiny])
assert tiny_acc==1. and tiny_loss<.02
# Mesma entrada, três classes: predição determinística é a mesma nas três linhas.
contradict_x=np.repeat(X[tiny[:1]],3,axis=0)
contradict_y=np.array([0,1,2])
contradict_loss,contradict_prob=ce(forward(pt,contradict_x)[0],contradict_y)
assert contradict_loss>=np.log(3)-1e-12
assert np.mean(contradict_prob.argmax(1)==contradict_y)==1/3
print("lote: loss inicial",tiny_initial,"final",tiny_loss,"acurácia",tiny_acc)
print("contradição: limite inferior CE =",np.log(3))


## 7. Métricas independentes do treinamento

Matriz C[real, previsto], recall por classe e macro-F1. Divisões por zero recebem zero
neste contrato, e o suporte é sempre mostrado. Isso não transforma uma classe ausente
em evidência de qualidade. O baseline constante prevê a classe mais frequente no treino.


In [ ]:
def metrics(labels,pred,c=3):
    cm=np.zeros((c,c),dtype=int)
    np.add.at(cm,(labels,pred),1)
    tp=np.diag(cm); support=cm.sum(1); predicted=cm.sum(0)
    precision=np.divide(tp,predicted,out=np.zeros(c),where=predicted>0)
    recall=np.divide(tp,support,out=np.zeros(c),where=support>0)
    f1=np.divide(2*tp,support+predicted,out=np.zeros(c),where=(support+predicted)>0)
    return {"cm":cm,"acc":float(tp.sum()/cm.sum()),"recall":recall,"precision":precision,"f1":f1,"macro_f1":float(f1.mean()),"support":support}

example=metrics(np.array([0,0,1,1,2,2]),np.array([0,1,1,1,0,2]))
assert np.array_equal(example["cm"],[[1,1,0],[0,2,0],[1,0,1]])
assert np.isclose(example["acc"],4/6)
assert np.allclose(example["recall"],[.5,1.,.5])
imbal=metrics(np.r_[np.zeros(90,dtype=int),np.ones(5,dtype=int),np.full(5,2)],np.zeros(100,dtype=int))
assert imbal["acc"]==.9 and imbal["recall"][1]==imbal["recall"][2]==0
print("baseline enganoso: acurácia",imbal["acc"],"recall",imbal["recall"],"macro-F1",imbal["macro_f1"])


## 8. Laço de treino com snapshots de validação

Cada época cobre todas as linhas, inclusive o último lote incompleto. Train CE e val CE
são medidas em eval, com os mesmos pesos ao fim da época e sem penalidade L2.
O objetivo regularizado continua sendo usado nas atualizações. Guardamos uma cópia
do menor val CE; o critério é estrito e empates preservam a primeira época.


In [ ]:
def train(config,epochs=120):
    p=init(SEED+5,h=16,ln=config["ln"])
    v={k:np.zeros_like(a) for k,a in p.items()}
    shuffle=np.random.default_rng(SEED+6); masks=np.random.default_rng(SEED+7)
    history=[]; best=None; best_loss=float("inf"); best_epoch=-1
    for epoch in range(epochs):
        order=shuffle.permutation(tr); seen=[]; norms=[]; ratios=[]; saturation=[]
        for start in range(0,len(order),37):
            ix=order[start:start+37]; seen.extend(ix.tolist())
            _,g,cache=loss_grad(p,X[ix],y[ix],config["lam"],True,config["q"],masks)
            norms.append([np.linalg.norm(g[k]) for k in ("W1","W2")])
            saturation.append(float(np.mean(np.abs(cache[1])>.99)))
            p,v,r=step(p,g,v,.03,.9); ratios.append([r["W1"],r["W2"]])
        assert np.array_equal(np.sort(seen),np.sort(tr))
        train_ce,_=ce(forward(p,X[tr])[0],y[tr])
        val_ce,_=ce(forward(p,X[va])[0],y[va])
        history.append([train_ce,val_ce,*np.mean(norms,0),*np.mean(ratios,0),np.mean(saturation)])
        if val_ce<best_loss:
            best_loss=val_ce; best_epoch=epoch+1; best=copy.deepcopy(p)
    return {"p":best,"history":np.array(history),"epoch":best_epoch,"val":best_loss,"config":config.copy()}

configs=[{"name":"mínima","ln":False,"q":1.,"lam":0.},
         {"name":"L2 + dropout","ln":False,"q":.9,"lam":.001},
         {"name":"LayerNorm","ln":True,"q":1.,"lam":0.}]
runs=[train(cfg) for cfg in configs]
selected=min(runs,key=lambda r:r["val"])
for r in runs: print(r["config"]["name"],"melhor época",r["epoch"],"val CE",r["val"])
assert np.isclose(selected["val"],ce(forward(selected["p"],X[va])[0],y[va])[0])
assert all(np.isfinite(r["history"]).all() for r in runs)
print("seleção exclusivamente pela validação:",selected["config"]["name"])


## 9. Curvas e gradientes por camada

Descrição textual: painel esquerdo mostra CE de treino e validação da configuração
selecionada, com a época restaurada indicada. Painel direito mostra a média das normas
dos gradientes de W1 e W2 por época. A média pode esconder picos; o código permite
substituí-la por máximo ou quantis ao investigar um problema específico.


In [ ]:
hist=selected["history"]
fig,axes=plt.subplots(1,2,figsize=(10,3.5))
ep=np.arange(1,len(hist)+1)
axes[0].plot(ep,hist[:,0],label="treino em eval")
axes[0].plot(ep,hist[:,1],label="validação")
axes[0].axvline(selected["epoch"],color="black",ls="--",label="snapshot escolhido")
axes[0].set(xlabel="época",ylabel="CE média",title="Curvas comparáveis")
for j,k in enumerate(("W1","W2")): axes[1].plot(ep,hist[:,2+j],label=k)
axes[1].set(xlabel="época",ylabel="norma L2 média do gradiente",title="Gradientes por camada")
for ax in axes: ax.legend(); ax.grid(alpha=.2)
fig.tight_layout(); plt.show(); plt.close(fig)
print("última época: razões de atualização W1/W2",hist[-1,4:6],"saturação tanh",hist[-1,6])


## 10. Análise de erros na validação

Os cinco erros de maior probabilidade prevista são candidatos à inspeção: IDs, classe
real, classe prevista e confiança. Não alteramos labels para concordar com o modelo.
Probabilidade softmax alta não é garantia de calibração.


In [ ]:
_,val_prob=ce(forward(selected["p"],X[va])[0],y[va])
val_pred=val_prob.argmax(1); val_metrics=metrics(y[va],val_pred)
wrong=np.flatnonzero(val_pred!=y[va])
rank=wrong[np.argsort(-val_prob[wrong].max(1))]
print("confusão na validação (linhas reais):",val_metrics["cm"],sep="\n")
for j in rank[:5]: print("ID",ids[va[j]],"real",y[va[j]],"previsto",val_pred[j],"confiança",float(val_prob[j].max()))
assert val_metrics["cm"].sum()==len(va)
assert all(y[va[j]]!=val_pred[j] for j in rank)


## 11. Teste reservado, uma vez após congelar a seleção

Este é o relatório final do protocolo predefinido. Não usamos este número para trocar
configuração, seed, número de épocas ou limiar. A comparação é ilustrativa de uma seed
e de dados sintéticos; não demonstra superioridade geral de um mecanismo.


In [ ]:
test_ce,test_prob=ce(forward(selected["p"],X[te])[0],y[te])
test_metrics=metrics(y[te],test_prob.argmax(1))
majority=np.bincount(y[tr]).argmax()
baseline=metrics(y[te],np.full(len(te),majority))
assert np.array_equal(forward(selected["p"],X[te])[0],forward(selected["p"],X[te])[0])
assert test_metrics["cm"].sum()==len(te)
print("teste CE",test_ce,"acurácia",test_metrics["acc"],"macro-F1",test_metrics["macro_f1"])
print("recall",test_metrics["recall"],"suporte",test_metrics["support"])
print("baseline constante: acurácia",baseline["acc"],"macro-F1",baseline["macro_f1"])


## 12. Artefato de inferência e auditoria final

Uma cópia do modelo, estatísticas de entrada, classes e configuração é suficiente para
reproduzir a inferência desta rede. Retomar o treinamento exige também velocidade do
momentum, estados dos RNGs, posição no lote e contador de passos. O snapshot de melhor
validação deste laboratório é de inferência; não é apresentado como checkpoint de retomada.


In [ ]:
artifact={"params":copy.deepcopy(selected["p"]),"mean":mu.copy(),"scale":scale.copy(),
          "classes":[0,1,2],"config":selected["config"].copy(),"epoch":selected["epoch"]}
reloaded=(Xraw[va]-artifact["mean"])/artifact["scale"]
assert np.array_equal(forward(artifact["params"],reloaded)[0],forward(selected["p"],X[va])[0])
checks={"splits disjuntos":set(tr).isdisjoint(te),"CE uniforme":abs(uniform-np.log(3))<1e-14,
        "labels inválidos rejeitados":rejected,"gradient checking":max(errors.values())<1e-7,
        "defeito detectado":bad_error>.1,"lote memorizado":tiny_acc==1. and tiny_loss<.02,
        "contradição reconhecida":contradict_loss>=np.log(3)-1e-12,
        "métrica manual":np.isclose(example["acc"],4/6),"curvas finitas":np.isfinite(hist).all(),
        "snapshot validado":np.isclose(selected["val"],hist[:,1].min()),
        "contagem teste":test_metrics["cm"].sum()==len(te),"inferência reproduzida":np.array_equal(reloaded,X[va])}
assert all(checks.values())
print(f"{sum(checks.values())}/{len(checks)} grupos de auditoria aprovados")


## Exercício de extensão

Injete um defeito por vez: troque labels entre linhas, remova `1-a*a`, atualize W2 antes
de calcular dW1 ou esqueça o modo eval do dropout. Registre qual teste falha primeiro.
Não ajuste hiperparâmetros para esconder a falha. Para cada correção, preserve um teste
que detecte a regressão.

Referências técnicas consultadas em 9 set. 2026:
- [Stanford CS231n — Learning](https://cs231n.github.io/neural-networks-3/).
- [Deep Learning — Practical Methodology](https://www.deeplearningbook.org/contents/guidelines.html).
- [D2L — Backpropagation](https://d2l.ai/chapter_multilayer-perceptrons/backprop.html).
- [NumPy — Random Generator](https://numpy.org/doc/stable/reference/random/generator.html).
